# Codelab: Pre-permissioning ADK Agent Identities on GCP Agent Engine

Welcome to this advanced enterprise codelab! This notebook demonstrates how to predict and pre-permission **Agent Identities** for ADK agents deployed to **Google Cloud's Agent Engine (Vertex AI Reasoning Engine)**.

## Conceptual Q&A: Your 201 Questions Answered

### Q1: Is it possible to specify an existing identity?
**Answer:** No. Google’s Agent Identity is designed to act as a cryptographically attested, SPIFFE-based identity that is intrinsically tied to the lifecycle and resource path of a specific agent. Because it is auto-provisioned by the platform (complete with a short-lived, auto-rotating X.509 certificate), you can't "bring your own" identity to the deployment.

### Q2: Is it possible to pre-create / pre-permission an identity?
**Answer:** **Yes!** You don't have to wait until after deployment to assign permissions. Because the identity is strictly tied to the deployment path, it is entirely predictable, meaning you can pre-permission it in IAM before the agent ever exists.

### Q3: Am I just approaching this problem wrong?
**Answer:** No, you are approaching it exactly right. Pre-permissioning using the predictable SPIFFE-based principal ID string is the industry-standard best-practice pattern for zero-downtime, secure-by-default enterprise agent setups.

## Code Walkthrough: Pre-permissioning in GCP

Let's programmatically execute these steps using Python and the Google Cloud CLI.

### Step 1: Discover Project Details
Let's retrieve the current Project ID and Project Number programmatically.

In [ ]:
import os
import subprocess

PROJECT_ID = os.environ.get('GOOGLE_CLOUD_PROJECT')
if not PROJECT_ID:
    try:
        PROJECT_ID = subprocess.check_output(
            ['gcloud', 'config', 'get-value', 'project']
        ).decode('utf-8').strip()
    except Exception:
        PROJECT_ID = 'your-gcp-project-id'

if not PROJECT_ID or PROJECT_ID == '(unset)':
    PROJECT_ID = 'your-gcp-project-id'

os.environ['CLOUDSDK_CORE_PROJECT'] = PROJECT_ID

# Retrieve project number using gcloud
try:
    project_number = subprocess.check_output(
        ['gcloud', 'projects', 'describe', PROJECT_ID, '--format=value(projectNumber)']
    ).decode('utf-8').strip()
except Exception:
    project_number = '123456789012'  # Placeholder project number

print(f"Project ID: {PROJECT_ID}")
print(f"Project Number: {project_number}")

### Step 2: Retrieve Organization ID
Let's retrieve the organization ID programmatically.

In [ ]:
import json

orgs_res = subprocess.check_output(
    ['gcloud', 'organizations', 'list', '--format=json']
).decode('utf-8')
orgs = json.loads(orgs_res)
org_id = orgs[0]['name'].split('/')[-1]

print(f"Organization ID: {org_id}")

### Step 3: Construct the Predictable Principal ID
Using the SPIFFE ID format, we can construct the principal ID for our agent. 

The format is:
`principal://agents.global.org-<ORG_ID>.system.id.goog/resources/aiplatform/projects/<PROJECT_NUMBER>/locations/<REGION>/reasoningEngines/<AGENT_NAME>`

In [ ]:
region = 'us-central1'
agent_name = 'my-adk-agent'

predicted_principal = f"principal://agents.global.org-{org_id}.system.id.goog/resources/aiplatform/projects/{project_number}/locations/{region}/reasoningEngines/{agent_name}"

print(f"Predicted Agent Identity Principal ID:\n{predicted_principal}")

### Step 4: Create a Secret in Google Secret Manager
Let's create a secret representing the credentials needed to log into your internal tool.

In [ ]:
from google.cloud import secretmanager

client = secretmanager.SecretManagerServiceClient()
secret_id = "internal-tool-credentials"
parent = f"projects/{PROJECT_ID}"

try:
    secret = client.create_secret(
        request={
            "parent": parent,
            "secret_id": secret_id,
            "secret": {"replication": {"automatic": {}}},
        }
    )
    print(f"Secret created successfully: {secret.name}")
except Exception as e:
    if "AlreadyExists" in str(e) or "already exists" in str(e):
        print(f"Secret '{secret_id}' already exists.")
    else:
        raise e

version = client.add_secret_version(
    request={
        "parent": f"{parent}/secrets/{secret_id}",
        "payload": {"data": b"my-super-secret-internal-password"},
    }
)
print(f"Added secret version: {version.name}")

### Step 5: Pre-Permission the Identity on Secret Manager
Now we will grant the predicted principal ID the **Secret Manager Secret Accessor** role (`roles/secretmanager.secretAccessor`) on the specific secret.

In [ ]:
print(f"Granting secretAccessor role to predicted principal ID...")

cmd_iam = [
    'gcloud', 'secrets', 'add-iam-policy-binding', secret_id,
    f'--member={predicted_principal}',
    '--role=roles/secretmanager.secretAccessor',
    f'--project={PROJECT_ID}',
    '--format=json'
]

subprocess.check_call(cmd_iam)
print("IAM Policy updated successfully!")

### Step 6: Verify IAM Policy Binding
Let's retrieve the IAM policy of the secret to prove that the predicted Agent Identity principal is successfully permissioned.

In [ ]:
iam_policy_json = subprocess.check_output([
    'gcloud', 'secrets', 'get-iam-policy', secret_id,
    f'--project={PROJECT_ID}',
    '--format=json'
]).decode('utf-8')

iam_policy = json.loads(iam_policy_json)
print("Current IAM Policy for Secret:")
print(json.dumps(iam_policy, indent=2))

### Step 7: Creating an Engine Without Agent Code (Identity-Only Provisioning)

In standard enterprise environments, you may want to provision the Agent Identity and establish all IAM permissions **before** your developer team finishes writing or deploying the actual agent code. 

By utilizing the Vertex AI `v1beta1` Python SDK client with `IdentityType.AGENT_IDENTITY` and omitting any agent code, you can create a lightweight, identity-only Agent Engine instance. This dynamically allocates the secure SPIFFE-based principal ID, which we can programmatically retrieve and use in IAM policy bindings.

In [ ]:
import vertexai
from vertexai import types

print("Initializing Vertex AI Client (v1beta1)...")
client = vertexai.Client(
    project=PROJECT_ID,
    location=region,
    http_options=dict(api_version="v1beta1")
)

print("Provisioning identity-only Agent Engine...")
remote_app = client.agent_engines.create(
    config={
        "display_name": "identity-only-setup",
        "identity_type": types.IdentityType.AGENT_IDENTITY,
    },
)

# Retrieve the newly created Agent Identity
effective_identity = remote_app.api_resource.spec.effective_identity
iam_principal = f"principal://{effective_identity}"

print("======================================================================")
print("Agent Identity Created Successfully!")
print("======================================================================")
print(f"SPIFFE ID:     {effective_identity}")
print(f"IAM Member ID: {iam_principal}")
print("======================================================================")

### Step 8: (Optional) Live Deployment Verification
If you have deployed a live agent (which has a system-assigned numeric ID), you can run this section to dynamically discover its active principal ID, grant it access, and verify the end-to-end integration!

In [ ]:
import json
import subprocess

try:
    print("Searching for live deployed 'Secret Agent' instances...")
    engines_res = subprocess.check_output([
        'gcloud', 'ai', 'reasoning-engines', 'list',
        f'--project={PROJECT_ID}',
        f'--region={region}',
        '--format=json'
    ]).decode('utf-8')
    engines = json.loads(engines_res)
    
    secret_agents = [e for e in engines if e.get('displayName') == 'Secret Agent']
    if secret_agents:
        live_resource_name = secret_agents[0]['name']
        live_id = live_resource_name.split('/')[-1]
        live_principal = f"principal://agents.global.org-{org_id}.system.id.goog/resources/aiplatform/projects/{project_number}/locations/{region}/reasoningEngines/{live_id}"
        print(f"Found Live Deployed Agent Name: {live_resource_name}")
        print(f"Constructed Live Principal ID: {live_principal}")
        
        # Grant permission to the live principal
        print("Granting secretAccessor role to live principal...")
        cmd_iam_live = [
            'gcloud', 'secrets', 'add-iam-policy-binding', secret_id,
            f'--member={live_principal}',
            '--role=roles/secretmanager.secretAccessor',
            f'--project={PROJECT_ID}',
            '--format=json'
        ]
        subprocess.check_call(cmd_iam_live)
        print("Live Agent Policy updated successfully!")
    else:
        print("No live 'Secret Agent' deployment found. Skipping live authorization.")
except Exception as e:
    print(f"Skipping live verification: {str(e)}")

## Conclusion

By predicting the SPIFFE-based principal ID, we successfully permissioned our agent **before** the agent exists or has been deployed. The moment your deployment finishes, the agent's identity will instantly inherit the `secretAccessor` role and can retrieve the credentials without any access-denied window!